# Functions & Monthly Adjustment

In [ ]:
# Run once in the LiaW_mf PowerShell environment if a package is missing:
# python -m pip install databricks-sql-connector pandas numpy openpyxl Jinja2

In [ ]:
# VS Code does not need Databricks %restart_python.
# Restart the notebook kernel from the VS Code toolbar when needed.

In [ ]:
import os
from pathlib import Path

import html
import numpy as np
import pandas as pd
from databricks import sql
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side,
)
from openpyxl.utils import get_column_letter


def query_databricks(query):
    """Run SQL in Databricks and return the result as a local pandas DataFrame."""
    required_variables = (
        "DATABRICKS_SERVER_HOSTNAME",
        "DATABRICKS_HTTP_PATH",
        "DATABRICKS_TOKEN",
    )
    missing = [name for name in required_variables if not os.getenv(name)]
    if missing:
        raise RuntimeError(
            "Missing environment variables: "
            + ", ".join(missing)
            + ". Start VS Code from the configured LiaW_mf PowerShell terminal."
        )

    with sql.connect(
        server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
        http_path=os.environ["DATABRICKS_HTTP_PATH"],
        access_token=os.environ["DATABRICKS_TOKEN"],
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            columns = [column[0] for column in cursor.description]

    return pd.DataFrame.from_records(rows, columns=columns)

## --Need Change--

### 1) NI Projects df with sos

In [0]:
# ============================================================
# Project与SOS Date Mapping
# ============================================================

project_sos_df = pd.DataFrame({
    "NI Project": [
        "H&S Nova",
        "H&S Proline", "Rej Ella Plus",
        "H&S Rose",
        "PNT Aurora JAR",
        'PNT 3MM GAIA SHM', 'PNT 3MM GAIA CON',
        'PNT 3MM GAIA Oil',
        "Rej Mermaid",
        "H&S TOP 5 variant upgrade", "PTN 3MM Diamond", "H&S  Core upgrade- LE Offline BLUEBELL & CAMELLIA", "RJ EC Ella DPB focus CAC",
        "H&S Headspa Sam's "
    ],

    "SOS Date": [
        "2025-12-01",
        "2025-12-22", "2025-12-22",
        "2026-01-19",
        "2026-01-26",
        "2026-03-23", "2026-03-23",
        "2026-05-06",
        "2026-07-01",
        "2026-07-06", "2026-07-06", "2026-07-06", "2026-07-06",
        "2026-07-23"
    ]
})


# 将SOS Date统一转成datetime
# 没有SOS Date的Project会变成NaT
project_sos_df["SOS Date"] = pd.to_datetime(
    project_sos_df["SOS Date"],
    errors="coerce"
)

In [0]:
project_sos_df

### 2) Current month & past fcst day

In [0]:
# 当前TG计算月份
current_month = "2026-07"

# 当前月份截至目前已经过去的FCST Day
# 请替换成你当前实际计算出来的current_pastday。
current_pastday = 19.5

cutoff_date = pd.Timestamp("2026-07-26")

### 3) fcst calendar

In [0]:
# ============================================================
# Monthly FCST Calendar
# ============================================================
# year-month：月份
# fcst_day：该月完整的FCST Working Day
# carry_in：该月Carry In Day
# carry_out：该月Carry Out Day
#
# TG计算主要使用：
# 1. fcst_day
# 2. carry_in
# ============================================================

fcst_wd_df = pd.DataFrame({
    "year_month": [
        # FY2526
        "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12",
        "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06",

        # FY2627
        "2026-07", "2026-08", "2026-09", "2026-10", "2026-11", "2026-12",
        "2027-01", "2027-02", "2027-03", "2027-04", "2027-05", "2027-06"
    ],

    "fcst_day": [
        # FY2526
        22.5, 22.5, 21.5, 19.0, 21.5, 21.5,
        22.0, 16.0, 22.0, 21.0, 20.5, 19.5,

        # FY2627
        23.0, 21.5, 21.5, 20.0, 20.5, 22.5,
        21.5, 16.0, 21.5, 21.0, 19.5, 20.5
    ],

    "carry_in": [
        # FY2526
        1.0, 1.5, 0.0, 1.5, 1.5, 0.0,
        1.5, 0.5, 1.5, 1.5, 1.5, 0.0,

        # FY2627
        1.5, 1.5, 1.0, 1.5, 0.5, 1.0,
        1.5, 0.0, 0.0, 1.5, 1.5, 1.0
    ],

    "carry_out": [
        # FY2526
        1.5, 0.0, 1.5, 1.5, 0.0, 1.5,
        0.5, 1.5, 1.5, 1.5, 0.0, 1.5,

        # FY2627
        1.5, 1.0, 1.5, 0.5, 1.0, 1.5,
        0.0, 0.0, 1.5, 1.5, 1.0, 1.5
    ]
})


# ============================================================
# 将YYYY-MM转换成Period
# ============================================================
# Period会同时比较年份和月份。
#
# 例如：2025-12 < 2026-07
#
# 所以不会出现只比较12和7导致跨年判断错误的问题。
# ============================================================

fcst_wd_df["month_period"] = pd.PeriodIndex(
    fcst_wd_df["year_month"],
    freq="M"
)

fcst_wd_df = (
    fcst_wd_df
    .sort_values("month_period")
    .reset_index(drop=True)
)

In [0]:
fcst_wd_df.head()

### 4) fcst day before sos date

In [0]:
# ============================================================
# SOS前已经过去的FCST Day Mapping
# ============================================================
#
# Key：SOS Date
# Value：该月月初至SOS前一天的普通FCST Day
#
# 重要：
# 1. 不包含SOS当天
# 2. 不包含Carry In
# 3. Carry In会在TG function中单独扣除
# ============================================================

# 请替换成FCST Calendar中的实际值
pre_sos_fcst_days = {
    pd.Timestamp("2025-12-01"): 0.0,

    pd.Timestamp("2025-12-22"): 15.0,

    # need verification
    pd.Timestamp("2026-01-19"): 11.0,

    # need verification
    pd.Timestamp("2026-01-26"): 16.0,

    # need verification
    pd.Timestamp("2026-03-23"): 15.0,

    # need verification
    pd.Timestamp("2026-05-06"): 2.0,

    pd.Timestamp("2026-07-01"): 0.0,

    pd.Timestamp("2026-07-06"): 3.0,

    pd.Timestamp("2026-07-23"): 16.0
}

In [0]:
pre_sos_fcst_days

### 5) Calculate TG for each project

#### calculate TG function

In [0]:
def calculate_project_tg(
    sos_date,
    current_month,
    current_pastday,
    fcst_calendar,
    pre_sos_days_map
):
    """
    根据Project SOS Date计算累计Time Gone（TG）。

    ==========================================================
    参数
    ==========================================================

    sos_date:
        Project的SOS Date。
        例如："2025-12-22"

    current_month:
        当前计算月份，格式为YYYY-MM。
        例如："2026-07"

    current_pastday:
        当前月份截至今天已经过去的FCST Day。
        例如：2026-07目前已经过去10.5个FCST Day，
        则current_pastday = 10.5。

    fcst_calendar:
        月度FCST Calendar，需要包含：
        - month_period
        - fcst_day
        - carry_in

    pre_sos_days_map:
        SOS Date之前已经过去的普通FCST Day Mapping。

    ==========================================================
    TG逻辑
    ==========================================================

    Case 1：SOS在月初，且当前月就是SOS月

        分子 = 当前月Past FCST Day

        分母 = 当前月Total FCST Day


    Case 2：SOS在月初，且当前月晚于SOS月

        分子 =
            SOS月完整FCST Day
            + 中间月份完整FCST Day
            + 当前月Past FCST Day

        分母 =
            SOS月完整FCST Day
            + 中间月份完整FCST Day
            + 当前月Total FCST Day


    Case 3：SOS在月中，且当前月就是SOS月

        分子 =
            当前月Past FCST Day
            - SOS之前的Past FCST Day
            - Carry In

        分母 =
            当前月Total FCST Day
            - SOS之前的Past FCST Day
            - Carry In


    Case 4：SOS在月中，且当前月晚于SOS月

        分子 =
            SOS月从SOS开始的有效FCST Day
            + 中间月份完整FCST Day
            + 当前月Past FCST Day

        分母 =
            SOS月从SOS开始的有效FCST Day
            + 中间月份完整FCST Day
            + 当前月Total FCST Day
    """

    # ========================================================
    # Rule 0：没有SOS Date
    # ========================================================
    # 如果Project没有SOS Date，则无法计算TG，返回NaN。
    # ========================================================

    if pd.isna(sos_date):
        return np.nan

    # 将SOS Date转换为标准Timestamp
    sos_date = pd.Timestamp(sos_date).normalize()

    # ========================================================
    # 转换成年月Period
    # ========================================================
    # 这里是YYYY-MM一起比较，不是只比较month number。
    #
    # 例如：
    # sos_period     = Period("2025-12")
    # current_period = Period("2026-07")
    #
    # 结果：
    # 2025-12 < 2026-07
    # ========================================================

    sos_period = sos_date.to_period("M")

    current_period = pd.Period(
        current_month,
        freq="M"
    )

    # ========================================================
    # Rule 1：SOS月份晚于当前月份
    # ========================================================
    # 例如：
    # SOS Month = 2026-08
    # Current Month = 2026-07
    #
    # Project尚未进入SOS月份，因此TG = 0。
    # ========================================================

    if sos_period > current_period:
        return 0.0

    # 复制Calendar，避免修改原DataFrame
    calendar = fcst_calendar.copy()

    # 如果Calendar还没有month_period，则自动创建
    if "month_period" not in calendar.columns:
        calendar["month_period"] = pd.PeriodIndex(
            calendar["year-month"],
            freq="M"
        )

    # 按月份排序
    calendar = calendar.sort_values(
        "month_period"
    )

    # ========================================================
    # 提取从SOS月份到当前月份的Calendar
    # ========================================================

    selected_months = calendar.loc[
        calendar["month_period"].between(
            sos_period,
            current_period
        )
    ].copy()

    # 如果Calendar找不到对应月份，返回NaN
    if selected_months.empty:
        return np.nan

    # ========================================================
    # 获取SOS月份的信息
    # ========================================================

    sos_month_row = selected_months.loc[
        selected_months["month_period"] == sos_period
    ]

    # Calendar中缺少SOS月份
    if sos_month_row.empty:
        return np.nan

    sos_month_row = sos_month_row.iloc[0]

    # SOS月份完整FCST Day
    sos_total_fcst_day = float(
        sos_month_row["fcst_day"]
    )

    # SOS月份Carry In
    sos_carry_in = float(
        sos_month_row["carry_in"]
    )

    # ========================================================
    # Rule 2：计算SOS月份从SOS开始可使用的FCST Day
    # ========================================================

    if sos_date.day == 1:

        # ----------------------------------------------------
        # SOS在月初
        # ----------------------------------------------------
        # SOS前没有任何FCST Day需要扣除。
        # 起始月使用完整FCST Day。
        # ----------------------------------------------------

        pre_sos_days = 0.0
        sos_effective_fcst_day = sos_total_fcst_day

    else:

        # ----------------------------------------------------
        # SOS在月中
        # ----------------------------------------------------
        # 从Mapping里读取SOS之前已经过去的普通FCST Day。
        # ----------------------------------------------------

        pre_sos_days = pre_sos_days_map.get(
            sos_date,
            np.nan
        )

        # 如果月中SOS没有维护pre_sos_fcst_days，
        # 则无法准确计算TG，返回NaN。
        if pd.isna(pre_sos_days):
            return np.nan

        # ----------------------------------------------------
        # SOS月份从SOS开始有效的FCST Day：
        #
        # Total FCST Day
        # - SOS之前的普通Past FCST Day
        # - Carry In
        # ----------------------------------------------------

        sos_effective_fcst_day = (
            sos_total_fcst_day
            - pre_sos_days
            - sos_carry_in
        )

        # 防止错误数据导致负数
        sos_effective_fcst_day = max(
            sos_effective_fcst_day,
            0.0
        )

    # ========================================================
    # Rule 3：当前月份就是SOS月份
    # ========================================================

    if current_period == sos_period:

        # ----------------------------------------------------
        # Case 3A：SOS在月初
        # ----------------------------------------------------

        if sos_date.day == 1:

            numerator = float(current_pastday)
            denominator = sos_total_fcst_day

        # ----------------------------------------------------
        # Case 3B：SOS在月中
        # ----------------------------------------------------

        else:

            # 分子：
            # 当前月已经过去的FCST Day
            # - SOS之前的Past FCST Day
            # - Carry In
            numerator = (
                float(current_pastday)
                - pre_sos_days
                - sos_carry_in
            )

            # 如果当前日期还没有走到SOS，
            # 计算可能为负数，此时TG分子按0处理。
            numerator = max(
                numerator,
                0.0
            )

            # 分母：
            # SOS月份从SOS开始有效的全部FCST Day
            denominator = sos_effective_fcst_day

    # ========================================================
    # Rule 4：当前月份晚于SOS月份
    # ========================================================

    else:

        # ----------------------------------------------------
        # 计算SOS月份和当前月份之间的所有完整月份
        # ----------------------------------------------------
        #
        # 例如：
        # SOS = 2025-12
        # Current Month = 2026-07
        #
        # 中间完整月份为：
        # 2026-01 至 2026-06
        #
        # 不包含SOS月份，也不包含当前月份。
        # ----------------------------------------------------

        middle_month_total = selected_months.loc[
            (
                selected_months["month_period"]
                > sos_period
            )
            &
            (
                selected_months["month_period"]
                < current_period
            ),
            "fcst_day"
        ].sum()

        middle_month_total = float(
            middle_month_total
        )

        # ----------------------------------------------------
        # 获取当前月份的完整FCST Day
        # ----------------------------------------------------

        current_month_row = selected_months.loc[
            selected_months["month_period"]
            == current_period
        ]

        # Calendar中没有当前月份
        if current_month_row.empty:
            return np.nan

        current_total_fcst_day = float(
            current_month_row["fcst_day"].iloc[0]
        )

        # ----------------------------------------------------
        # 分子
        # ----------------------------------------------------
        #
        # SOS月份从SOS开始的有效FCST Day
        # + 中间月份完整FCST Day
        # + 当前月份已经过去的FCST Day
        # ----------------------------------------------------

        numerator = (
            sos_effective_fcst_day
            + middle_month_total
            + float(current_pastday)
        )

        # ----------------------------------------------------
        # 分母
        # ----------------------------------------------------
        #
        # SOS月份从SOS开始的有效FCST Day
        # + 中间月份完整FCST Day
        # + 当前月份完整FCST Day
        # ----------------------------------------------------

        denominator = (
            sos_effective_fcst_day
            + middle_month_total
            + current_total_fcst_day
        )

    # ========================================================
    # 最终TG
    # ========================================================

    # 如果分母为0或负数，则无法计算
    if denominator <= 0:
        return np.nan

    tg = numerator / denominator

    # 防止异常数据导致TG小于0%或超过100%
    tg = np.clip(tg, 0, 1)

    return float(tg)

#### calculate project TG

In [0]:
# ============================================================
# 对每一个Project，根据各自SOS Date计算TG
# ============================================================

project_sos_df["TG"] = project_sos_df["SOS Date"].apply(
    lambda sos_date: calculate_project_tg(
        sos_date=sos_date,
        current_month=current_month,
        current_pastday=current_pastday,
        fcst_calendar=fcst_wd_df,
        pre_sos_days_map=pre_sos_fcst_days
    )
)

# ============================================================
# 将数值TG转换为百分比显示
#
# Time Gone保留数值，可以继续用于计算。
# Time Gone %用于Table/display。
# ============================================================

project_sos_df["TG%"] = (
    project_sos_df["TG"]
    .apply(
        lambda x: f"{x:.0%}"
        if pd.notna(x)
        else ""
    )
)

In [0]:
project_sos_df.dtypes

#### Verify final TG

In [0]:
# ============================================================
# 检查每个Project对应的SOS前FCST Day
# ============================================================
def get_tg_status(sos_date):
    """
    检查TG能否正常计算。
    """

    if pd.isna(sos_date):
        return "Missing SOS Date"

    sos_date = pd.Timestamp(sos_date).normalize()

    # 月初SOS不需要额外维护Pre SOS Day
    if sos_date.day == 1:
        return "Ready - Month Start SOS"

    # 月中SOS需要在Mapping中维护
    if sos_date not in pre_sos_fcst_days:
        return "Missing Pre SOS FCST Day"

    return "Ready - Mid Month SOS"


project_sos_df["TG Status"] = (
    project_sos_df["SOS Date"]
    .apply(get_tg_status)
)

In [0]:
project_sos_df

# --Read-in Data--

## shipment & md

In [ ]:
# The join and aggregation run in Databricks; only the summarized result is
# transferred to this local notebook.
duplicate_fpc_query = """
SELECT fpc, COUNT(*) AS count
FROM bigu_ps_sandbox.ps_demand_bronze.lia_hc_md_transformed
GROUP BY fpc
HAVING COUNT(*) > 1
ORDER BY count DESC, fpc
"""

shipment_query = f"""
WITH joined_table AS (
    SELECT
        s.trans_date,
        s.fpc_code,
        s.channel,
        h.csu_ind,
        h.level_2,
        h.level_ni,
        s.giv_for_planning_decomposed,
        s.giv_mtd_oo_decomposed,
        s.giv_yo_oo_decomposed
    FROM bigu_ps_sandbox.ps_demand_cpf.dp_shipment AS s
    INNER JOIN bigu_ps_sandbox.ps_demand_bronze.lia_hc_md_transformed AS h
        ON CAST(s.fpc_code AS STRING) = CAST(h.fpc AS STRING)
    WHERE s.trans_date <= DATE '{cutoff_date:%Y-%m-%d}'
),
channel_summary AS (
    SELECT
        trans_date,
        fpc_code AS fpc,
        first(csu_ind, true) AS csu_ind,
        first(level_2, true) AS brand,
        first(level_ni, true) AS ni_project,
        channel,
        SUM(COALESCE(giv_for_planning_decomposed, 0))
            AS giv_for_planning_decomposed,
        SUM(COALESCE(giv_mtd_oo_decomposed, 0))
            AS giv_mtd_oo_decomposed,
        SUM(COALESCE(giv_yo_oo_decomposed, 0))
            AS giv_yo_oo_decomposed
    FROM joined_table
    GROUP BY trans_date, fpc_code, channel
),
total_summary AS (
    SELECT
        trans_date,
        fpc_code AS fpc,
        first(csu_ind, true) AS csu_ind,
        first(level_2, true) AS brand,
        first(level_ni, true) AS ni_project,
        'Total' AS channel,
        SUM(COALESCE(giv_for_planning_decomposed, 0))
            AS giv_for_planning_decomposed,
        SUM(COALESCE(giv_mtd_oo_decomposed, 0))
            AS giv_mtd_oo_decomposed,
        SUM(COALESCE(giv_yo_oo_decomposed, 0))
            AS giv_yo_oo_decomposed
    FROM joined_table
    GROUP BY trans_date, fpc_code
)
SELECT * FROM channel_summary
UNION ALL
SELECT * FROM total_summary
"""

duplicate_fpc_check = query_databricks(duplicate_fpc_query)
shipment_df = query_databricks(shipment_query)

## Join tables

In [ ]:
# Databricks completed the filtering, join, and aggregation in shipment_query.
display(shipment_df.head(100))

In [ ]:
# Check whether the HC master data has duplicate FPC values.
display(duplicate_fpc_check)

In [ ]:
# Local pandas result returned by Databricks SQL.
shipment_df.head()

In [ ]:
print("Columns:", shipment_df.columns.tolist())
print(f"Summarized rows: {len(shipment_df):,}")
display(shipment_df.dtypes)

In [ ]:
# Channel and Total summaries are calculated remotely in shipment_query.

In [ ]:
shipment_df_display = (
    shipment_df.assign(
        _channel_sort=np.where(shipment_df["channel"].eq("Total"), 1, 0)
    )
    .sort_values(["trans_date", "fpc", "_channel_sort", "channel"])
    .drop(columns="_channel_sort")
)
display(shipment_df_display.head(100))

In [ ]:
print("DataFrame type:", type(shipment_df))
print("DataFrame shape:", shipment_df.shape)

In [0]:
shipment_oo_df = shipment_df.copy()

#### checkpoint

In [0]:
# 确保trans_date是datetime
shipment_oo_df["trans_date"] = pd.to_datetime(
    shipment_oo_df["trans_date"],
    errors="coerce"
)

cutoff_date = pd.Timestamp("2026-07-26")

shipment_oo_df = shipment_oo_df.loc[
    shipment_oo_df["trans_date"] <= cutoff_date
].copy()

In [0]:
shipment_oo_df["trans_date"].max()

In [0]:
shipment_oo_df["year_month"] = (
    pd.to_datetime(
        shipment_oo_df["trans_date"]
    )
    .dt.strftime("%Y-%m")
)

In [0]:
# ============================================================
# 将GIV列转换成Numeric
# ============================================================
giv_columns = [
    "giv_for_planning_decomposed",
    "giv_mtd_oo_decomposed",
    "giv_yo_oo_decomposed"
]

for col in giv_columns:
    shipment_oo_df[col] = (
        pd.to_numeric(
            shipment_oo_df[col],
            errors="coerce"
        )
        .fillna(0)
    )



In [0]:
shipment_oo_df.head()

In [0]:
# 查看每一列的数据类型
shipment_oo_df.dtypes

In [0]:
rose_july = shipment_oo_df.loc[
    (shipment_oo_df["ni_project"] == "H&S Rose")
    & (
        pd.to_datetime(
            shipment_oo_df["trans_date"]
        ).dt.strftime("%Y-%m")
        == "2026-07"
    )
].copy()

display(rose_july)

## Pre-check for MTD ship+oo

In [0]:
def check_project_giv_oo(
    project_name,
    month,
    shipment_oo_df
):

    result = (
        shipment_oo_df.loc[
            (shipment_oo_df["year_month"] == month)
            & (shipment_oo_df["ni_project"] == project_name)
        ]
        .groupby(
            [
                "channel",
                #"fpc"
            ],
            as_index=False
        )
        .agg(
            giv=(
                "giv_for_planning_decomposed",
                "sum"
            ),
            oo=(
                #"giv_mtd_oo_decomposed",
                "giv_yo_oo_decomposed",
                "sum"
            )
        )
    )

    # MM
    result["giv"] = result["giv"] / 1_000_000
    result["oo"] = result["oo"] / 1_000_000

    result["giv+oo"] = (
        result["giv"]
        + result["oo"]
    )

    result[
        ["giv", "oo", "giv+oo"]
    ] = (
        result[
            ["giv", "oo", "giv+oo"]
        ]
        .round(1)
    )

    # Channel排序
    channel_order = {
        "eCom": 1,
        "eCom B2C": 1,
        "Dcom": 2,
        "O&O": 3,
        "D-Partner": 4,
        "Total": 99
    }

    result["_sort"] = (
        result["channel"]
        .astype(str)
        .str.strip()
        .map(channel_order)
        .fillna(50)
    )

    result = (
        result
        .sort_values(
            #["_sort", "fpc"]
            "_sort"
        )
        .drop(columns="_sort")
        .reset_index(drop=True)
    )

    return result

In [0]:
check_project_giv_oo(
  project_name="PNT 3MM GAIA SHM",
  month="2026-07",
  shipment_oo_df=shipment_oo_df
    )

In [0]:
shipment_oo_df.ni_project.unique()

In [0]:
shipment_oo_df.loc[
    (shipment_oo_df["year_month"] == "2026-07")
    & (shipment_oo_df["ni_project"] == "H&S Nova")
]["fpc"].unique()

In [0]:
check_codes = ['80889183', '80889184', '80889185', '80889182']

check_df = (
    shipment_oo_df.loc[
        (
            pd.to_datetime(
                shipment_oo_df["trans_date"]
            ).dt.strftime("%Y-%m") == "2026-07"
        )
        &
        (
            shipment_oo_df["fpc"]
            .astype(str)
            .isin(check_codes)
        )
        &
        (
            shipment_oo_df["channel"] != "Total"
        )
    ]
    .groupby(
        [
            "fpc",
            "ni_project"
        ],
        as_index=False
    )
    .agg(
        giv=(
            "giv_for_planning_decomposed",
            "sum"
        ),
        oo=(
            "giv_mtd_oo_decomposed",
            "sum"
        )
    )
)

# 转MM
check_df["giv"] = check_df["giv"] / 1_000_000
check_df["oo"] = check_df["oo"] / 1_000_000

check_df["giv+oo"] = (
    check_df["giv"]
    + check_df["oo"]
)

check_df[
    ["giv", "oo", "giv+oo"]
] = (
    check_df[
        ["giv", "oo", "giv+oo"]
    ]
    .round(1)
)

display(
    check_df.sort_values("fpc")
)

## LPA data df

In [ ]:
# Read the LPA table from Databricks into local pandas.
booklet_df = query_databricks(
    """
    SELECT *
    FROM bigu_ps_sandbox.ps_demand_bronze.Lia_hc_ni_lpa2627
    """
)

booklet_df["year_month"] = (
    pd.to_datetime(booklet_df["year_month"], errors="coerce")
    .dt.strftime("%Y-%m")
)

In [ ]:
print(booklet_df.shape)
booklet_df.head()

In [0]:
booklet_df.dtypes

# --Create tracking table--

In [0]:
# project_sos_df
# shipment_oo_df
# booklet_df

In [0]:
project_sos_df.head()
print(project_sos_df["NI Project"].unique())

In [0]:
print(shipment_oo_df.ni_project.unique())
shipment_oo_df.head()

In [0]:
print(booklet_df.project_name.unique())
booklet_df.head()

## MTD

In [0]:
import pandas as pd
import numpy as np


def create_mtd_table(current_month, current_pastday):
    """
    创建指定月份的MTD Tracking Table。

    Parameters
    ----------
    current_month : str
        当前月份，格式为yyyy-mm，例如"2026-07"。

    current_pastday : float
        当前月份已经过去的FCST Day，例如17.5。

    Global DataFrames
    -----------------
    fcst_wd_df
    booklet_df
    shipment_oo_df
    project_sos_df

    Returns
    -------
    mtd_table_output :
        最终展示宽表。

    mtd_table_long :
        Project × Channel粒度的numeric明细表。
    """

    # ========================================================
    # 0. Project名称标准化
    # ========================================================

    def clean_project_name(series):
        """
        清理Project Name中的：
        - HTML不换行空格
        - Unicode不换行空格
        - 连续空格
        - 首尾空格
        - 特殊撇号
        """

        return (
            series
            .astype("string")
            .str.replace("&nbsp;", " ", regex=False)
            .str.replace("\u00a0", " ", regex=False)
            .str.replace("’", "'", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )


    def standardize_names(df):
        """
        统一Project Name和Brand。
        """

        df = df.copy()

        # ----------------------------------------------------
        # Project Name标准化
        # ----------------------------------------------------

        if "ni_project" in df.columns:

            df["ni_project"] = clean_project_name(
                df["ni_project"]
            )

            project_mapping = {
                # H&S Core Offline旧名称
                "H&S Core upgrade- LE Offline 清柚蓝风铃 & 乌木山茶":
                    "H&S Core upgrade- LE Offline BLUEBELL & CAMELLIA",

                "H&S Core LE 清柚蓝风铃 & 乌木山茶":
                    "H&S Core upgrade- LE Offline BLUEBELL & CAMELLIA",

                "H&S Core upgrade- LE Offline BLUEBELL & CAMELLIA":
                    "H&S Core upgrade- LE Offline BLUEBELL & CAMELLIA",

                # H&S Ironman
                "H&S Ironman TMS 白檀晚香玉":
                    "H&S Ironman TMS YULAN",

                "H&S Ironman PDD 清茉玉兰":
                    "H&S Ironman PDD Yulan",

                # H&S Core Online
                "H&S Core upgrade- LE Online 清新海盐&春日樱花":
                    "H&S Core upgrade- LE Online Sea Salt & Sakura",

                "H&S Core upgrade- LE Online 清新海盐 & 春日樱花":
                    "H&S Core upgrade- LE Online Sea Salt & Sakura",

                # HR
                "HR TMFS 妃子一笑":
                    "HR TMFS Kirei LE",

                "HR TMFS 松池":
                    "HR TMFS LE 2.0 Pine Pond",

                "HR TMFS 松池 枕边喷雾":
                    "HR TMFS LE 2.0 Pine Pond Pillow Mist",

                # PTN其他项目
                "PTN 3MM DY 雨后森林":
                    "PTN 3MM DY Serene Forest",

                "PTN 3MM PDD 荔枝雪松":
                    "PTN 3MM PDD Fra 1 Lychee Cedarwood",

                # GAIA SHM + CON统一名称
                "PNT 3MM GAIA CON":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA SHM":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA - CON":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA - SHM":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA - SHM & CON":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA - SHM&CON":
                    "PNT 3MM GAIA SHM&CON",

                "PNT 3MM GAIA SHM & CON":
                    "PNT 3MM GAIA SHM&CON"
            }

            df["ni_project"] = (
                df["ni_project"]
                .replace(project_mapping)
            )

        # ----------------------------------------------------
        # Brand标准化
        # ----------------------------------------------------

        if "brand" in df.columns:

            df["brand"] = (
                df["brand"]
                .astype("string")
                .str.replace("&nbsp;", " ", regex=False)
                .str.replace("\u00a0", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
                .replace("", pd.NA)
            )

            # 所有Rej开头的Brand统一为REJ
            rej_mask = (
                df["brand"]
                .fillna("")
                .str.lower()
                .str.startswith("rej")
            )

            df.loc[rej_mask, "brand"] = "REJ"

            # PNT / PTN统一为PTN
            ptn_mask = (
                df["brand"]
                .fillna("")
                .str.upper()
                .isin(["PNT", "PTN"])
            )

            df.loc[ptn_mask, "brand"] = "PTN"

        return df


    def create_project_key(series):
        """
        创建Merge专用Project Key。

        显示名称保持原样；
        Merge使用大写、清理空格后的Key。
        """

        return (
            clean_project_name(series)
            .str.upper()
        )


    # ========================================================
    # 1. 当前月FCST Day与Common TG
    # ========================================================

    current_month = str(current_month).strip()
    current_pastday = float(current_pastday)

    fcst_calendar = fcst_wd_df.copy()

    # 兼容year-month和year_month两种列名
    if (
        "year-month" in fcst_calendar.columns
        and "year_month" not in fcst_calendar.columns
    ):
        fcst_calendar = fcst_calendar.rename(
            columns={
                "year-month": "year_month"
            }
        )

    fcst_calendar["year_month"] = (
        fcst_calendar["year_month"]
        .astype(str)
        .str.strip()
    )

    current_fcst_match = fcst_calendar.loc[
        fcst_calendar["year_month"] == current_month,
        "fcst_day"
    ]

    if current_fcst_match.empty:
        raise ValueError(
            f"{current_month}在fcst_wd_df中没有对应的fcst_day"
        )

    current_total_fcst_day = float(
        current_fcst_match.iloc[0]
    )

    if current_total_fcst_day <= 0:
        raise ValueError(
            "Current month total FCST Day必须大于0"
        )

    # Past Day限制在0至Total FCST Day之间
    current_pastday = min(
        max(current_pastday, 0),
        current_total_fcst_day
    )

    common_tg = (
        current_pastday
        / current_total_fcst_day
    )


    # ========================================================
    # 2. 判断月中还是月末
    # ========================================================

    is_actualized = np.isclose(
        current_pastday,
        current_total_fcst_day
    )

    if is_actualized:

        tracking_type = "actualized"
        actual_column_name = "MTD"

    else:

        tracking_type = "mid_month"
        actual_column_name = "MTD + OO"


    # ========================================================
    # 3. 整理Project SOS清单
    # ========================================================
    # 最终明细Project只能来自这张表。
    # ========================================================

    project_tg = (
        project_sos_df[
            [
                "NI Project",
                "SOS Date",
                "TG"
            ]
        ]
        .copy()
        .rename(
            columns={
                "NI Project": "ni_project",
                "SOS Date": "sos_date",
                "TG": "project_specific_tg"
            }
        )
    )

    project_tg = standardize_names(
        project_tg
    )

    project_tg["project_key"] = (
        create_project_key(
            project_tg["ni_project"]
        )
    )

    project_tg["sos_date"] = pd.to_datetime(
        project_tg["sos_date"],
        errors="coerce"
    )

    project_tg["project_specific_tg"] = (
        pd.to_numeric(
            project_tg["project_specific_tg"],
            errors="coerce"
        )
    )

    # 名称标准化后可能有重复Project，例如GAIA SHM/CON
    project_tg = (
        project_tg
        .sort_values(
            [
                "project_key",
                "sos_date"
            ],
            na_position="last"
        )
        .groupby(
            "project_key",
            as_index=False,
            dropna=False
        )
        .agg(
            ni_project=(
                "ni_project",
                "first"
            ),
            sos_date=(
                "sos_date",
                "min"
            ),
            project_specific_tg=(
                "project_specific_tg",
                "first"
            )
        )
    )

    allowed_project_keys = set(
        project_tg["project_key"]
        .dropna()
        .unique()
    )

    current_period = pd.Period(
        current_month,
        freq="M"
    )


    # ========================================================
    # 4. 决定每个Project最终TG
    # ========================================================

    def select_final_tg(row):

        sos_date = row["sos_date"]

        if pd.isna(sos_date):
            return np.nan

        sos_period = sos_date.to_period("M")

        # 未来SOS
        if sos_period > current_period:
            return 0.0

        # 前几个月已经SOS
        if sos_period < current_period:
            return common_tg

        # 当前月1号SOS
        if sos_date.day == 1:
            return common_tg

        # 当前月月中SOS
        return row["project_specific_tg"]


    project_tg["final_tg"] = (
        project_tg.apply(
            select_final_tg,
            axis=1
        )
    )


    # ========================================================
    # 5. 整理Shipment
    # ========================================================

    shipment = shipment_oo_df.copy()

    # year_month处理
    if "year_month" not in shipment.columns:

        if "trans_date" not in shipment.columns:
            raise KeyError(
                "shipment_oo_df中缺少year_month和trans_date"
            )

        shipment["year_month"] = (
            pd.to_datetime(
                shipment["trans_date"],
                errors="coerce"
            )
            .dt.strftime("%Y-%m")
        )

    else:

        shipment["year_month"] = (
            pd.to_datetime(
                shipment["year_month"],
                errors="coerce"
            )
            .dt.strftime("%Y-%m")
        )

    shipment = standardize_names(
        shipment
    )

    shipment["project_key"] = (
        create_project_key(
            shipment["ni_project"]
        )
    )


    # ========================================================
    # 6. GIV和OO转成MM
    # ========================================================
    # 如果shipment_oo_df已经是MM，
    # 删除下面的 / 1_000_000。
    # ========================================================

    shipment_value_columns = [
        "giv_for_planning_decomposed",
        "giv_yo_oo_decomposed"
    ]

    for col in shipment_value_columns:

        if col not in shipment.columns:
            raise KeyError(
                f"shipment_oo_df中缺少字段：{col}"
            )

        shipment[col] = (
            pd.to_numeric(
                shipment[col],
                errors="coerce"
            )
            .fillna(0)
            / 1_000_000
        )


    # ========================================================
    # 7. 计算Actual Value
    # ========================================================

    if tracking_type == "mid_month":

        shipment["actual_value"] = (
            shipment["giv_for_planning_decomposed"]
            + shipment["giv_yo_oo_decomposed"]
        )

    else:

        shipment["actual_value"] = (
            shipment["giv_for_planning_decomposed"]
        )


    # 只保留当前月份及SOS Project
    shipment = shipment.loc[
        (shipment["year_month"] == current_month)
        & shipment["project_key"].isin(
            allowed_project_keys
        )
    ].copy()


    # ========================================================
    # 8. 标准化Shipment Channel
    # ========================================================

    shipment["channel"] = (
        shipment["channel"]
        .astype("string")
        .str.strip()
    )

    channel_mapping = {
        "eCom": "eCom B2C",
        "ecom": "eCom B2C",
        "B2C": "eCom B2C",
        "eCom B2C": "eCom B2C",

        "Dcom": "Dcom",
        "dcom": "Dcom",

        "o_o": "O&O",
        "O&O": "O&O",

        "dpb": "D-Partner",
        "d partner": "D-Partner",
        "D Partner": "D-Partner",
        "D-Partner": "D-Partner",

        "Total": "Total",
        "total": "Total"
    }

    shipment["channel"] = (
        shipment["channel"]
        .replace(channel_mapping)
    )


    # ========================================================
    # 9. Shipment按Project Key × Channel汇总
    # ========================================================

    shipment_project_channel = (
        shipment
        .groupby(
            [
                "year_month",
                "project_key",
                "brand",
                "channel"
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            actual_value=(
                "actual_value",
                "sum"
            )
        )
    )


    # ========================================================
    # 10. Project → Brand Mapping
    # ========================================================

    project_brand_mapping = (
        shipment_project_channel[
            [
                "project_key",
                "brand"
            ]
        ]
        .dropna(
            subset=[
                "project_key",
                "brand"
            ]
        )
        .drop_duplicates(
            subset=["project_key"],
            keep="first"
        )
        .rename(
            columns={
                "brand": "mapped_brand"
            }
        )
    )


    # ========================================================
    # 11. 整理Booklet
    # ========================================================

    booklet = booklet_df.copy()

    booklet["year_month"] = (
        pd.to_datetime(
            booklet["year_month"],
            errors="coerce"
        )
        .dt.strftime("%Y-%m")
    )

    booklet = booklet.loc[
        booklet["year_month"] == current_month
    ].copy()

    booklet_long = booklet.melt(
        id_vars=[
            "project_name",
            "year_month"
        ],
        value_vars=[
            "ecom",
            "dcom",
            "o_o",
            "dpb",
            "total"
        ],
        var_name="source_channel",
        value_name="booklet"
    )

    booklet_channel_mapping = {
        "ecom": "eCom B2C",
        "dcom": "Dcom",
        "o_o": "O&O",
        "dpb": "D-Partner",
        "total": "Total"
    }

    booklet_long["channel"] = (
        booklet_long["source_channel"]
        .map(booklet_channel_mapping)
    )

    booklet_long = (
        booklet_long
        .rename(
            columns={
                "project_name": "ni_project"
            }
        )
        [
            [
                "year_month",
                "ni_project",
                "channel",
                "booklet"
            ]
        ]
    )

    booklet_long = standardize_names(
        booklet_long
    )

    booklet_long["project_key"] = (
        create_project_key(
            booklet_long["ni_project"]
        )
    )

    booklet_long["booklet"] = (
        pd.to_numeric(
            booklet_long["booklet"],
            errors="coerce"
        )
    )

    # 只保留SOS Project
    booklet_long = booklet_long.loc[
        booklet_long["project_key"]
        .isin(allowed_project_keys)
    ].copy()

    # GAIA改名后重新合并
    booklet_long = (
        booklet_long
        .groupby(
            [
                "year_month",
                "project_key",
                "channel"
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            booklet=(
                "booklet",
                lambda x: x.sum(min_count=1)
            )
        )
    )


    # ========================================================
    # 12. Booklet匹配检查
    # ========================================================

    booklet_keys = set(
        booklet_long["project_key"]
        .dropna()
        .unique()
    )

    missing_booklet_keys = (
        allowed_project_keys
        - booklet_keys
    )

    if missing_booklet_keys:

        missing_booklet_projects = (
            project_tg.loc[
                project_tg["project_key"]
                .isin(missing_booklet_keys),
                "ni_project"
            ]
            .tolist()
        )

        print(
            "SOS Projects not matched in Booklet:"
        )

        for project in missing_booklet_projects:
            print(repr(project))


    # ========================================================
    # 13. Shipment匹配检查
    # ========================================================

    shipment_keys = set(
        shipment_project_channel["project_key"]
        .dropna()
        .unique()
    )

    missing_shipment_keys = (
        allowed_project_keys
        - shipment_keys
    )

    if missing_shipment_keys:

        missing_shipment_projects = (
            project_tg.loc[
                project_tg["project_key"]
                .isin(missing_shipment_keys),
                "ni_project"
            ]
            .tolist()
        )

        print(
            "SOS Projects not matched in Shipment:"
        )

        for project in missing_shipment_projects:
            print(repr(project))


    # ========================================================
    # 14. 建立SOS Project × Channel骨架
    # ========================================================

    standard_channels = [
        "eCom B2C",
        "Dcom",
        "O&O",
        "D-Partner",
        "Total"
    ]

    project_base = (
        project_tg[
            [
                "project_key",
                "ni_project",
                "sos_date",
                "final_tg"
            ]
        ]
        .drop_duplicates(
            subset=["project_key"]
        )
        .copy()
    )

    project_base["_join_key"] = 1

    channel_base = pd.DataFrame({
        "channel": standard_channels,
        "_join_key": 1
    })

    project_channel_base = (
        project_base
        .merge(
            channel_base,
            on="_join_key",
            how="inner"
        )
        .drop(
            columns="_join_key"
        )
    )

    project_channel_base["year_month"] = (
        current_month
    )


    # ========================================================
    # 15. 使用Project Key合并Booklet和Shipment
    # ========================================================

    project_detail_long = (
        project_channel_base
        .merge(
            booklet_long[
                [
                    "year_month",
                    "project_key",
                    "channel",
                    "booklet"
                ]
            ],
            on=[
                "year_month",
                "project_key",
                "channel"
            ],
            how="left"
        )
        .merge(
            shipment_project_channel[
                [
                    "year_month",
                    "project_key",
                    "brand",
                    "channel",
                    "actual_value"
                ]
            ],
            on=[
                "year_month",
                "project_key",
                "channel"
            ],
            how="left"
        )
    )


    # ========================================================
    # 16. 补充Brand
    # ========================================================

    project_detail_long = (
        project_detail_long
        .merge(
            project_brand_mapping,
            on="project_key",
            how="left"
        )
    )

    project_detail_long["brand"] = (
        project_detail_long["brand"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .fillna(
            project_detail_long["mapped_brand"]
        )
    )

    project_detail_long = (
        project_detail_long
        .drop(
            columns=["mapped_brand"]
        )
    )

    project_detail_long = standardize_names(
        project_detail_long
    )


    # 如果Shipment没有数据，根据Project Name推导Brand
    missing_brand = (
        project_detail_long["brand"].isna()
    )

    project_upper = (
        project_detail_long["ni_project"]
        .astype(str)
        .str.upper()
    )

    project_detail_long.loc[
        missing_brand
        & project_upper.str.startswith("H&S"),
        "brand"
    ] = "H&S"

    project_detail_long.loc[
        missing_brand
        & project_upper.str.startswith(
            ("PNT", "PTN")
        ),
        "brand"
    ] = "PTN"

    project_detail_long.loc[
        missing_brand
        & project_upper.str.startswith(
            ("REJ", "RJ ")
        ),
        "brand"
    ] = "REJ"

    project_detail_long["brand"] = (
        project_detail_long["brand"]
        .fillna("Other")
    )


    # ========================================================
    # 17. 数值处理
    # ========================================================

    project_detail_long["booklet"] = (
        pd.to_numeric(
            project_detail_long["booklet"],
            errors="coerce"
        )
    )

    project_detail_long["actual_value"] = (
        pd.to_numeric(
            project_detail_long["actual_value"],
            errors="coerce"
        )
        .fillna(0)
    )


    # ========================================================
    # 18. Project级别vs. Booklet TG
    # ========================================================

    valid_project = (
        project_detail_long["booklet"].notna()
        & (project_detail_long["booklet"] != 0)
        & project_detail_long["final_tg"].notna()
    )

    project_detail_long["vs_booklet_tg"] = (
        np.where(
            valid_project,
            (
                project_detail_long["actual_value"]
                / project_detail_long["booklet"]
                - project_detail_long["final_tg"]
            ),
            np.nan
        )
    )


    # ========================================================
    # 19. 创建Subtotal / Total Function
    # ========================================================

    def create_summary_rows(
        source_df,
        summary_brand,
        summary_project
    ):
        """
        按Channel合计Booklet和Actual。

        汇总行TG使用Common TG。
        """

        if source_df.empty:

            return pd.DataFrame(
                columns=[
                    "year_month",
                    "brand",
                    "ni_project",
                    "project_key",
                    "channel",
                    "booklet",
                    "actual_value",
                    "sos_date",
                    "final_tg",
                    "vs_booklet_tg"
                ]
            )

        summary = (
            source_df
            .groupby(
                [
                    "year_month",
                    "channel"
                ],
                as_index=False,
                dropna=False
            )
            .agg(
                booklet=(
                    "booklet",
                    lambda x: x.sum(min_count=1)
                ),
                actual_value=(
                    "actual_value",
                    "sum"
                )
            )
        )

        summary["brand"] = summary_brand
        summary["ni_project"] = summary_project
        summary["project_key"] = (
            summary_project.upper()
        )
        summary["sos_date"] = pd.NaT
        summary["final_tg"] = common_tg

        valid_summary = (
            summary["booklet"].notna()
            & (summary["booklet"] != 0)
        )

        summary["vs_booklet_tg"] = np.where(
            valid_summary,
            (
                summary["actual_value"]
                / summary["booklet"]
                - common_tg
            ),
            np.nan
        )

        return summary[
            [
                "year_month",
                "brand",
                "ni_project",
                "project_key",
                "channel",
                "booklet",
                "actual_value",
                "sos_date",
                "final_tg",
                "vs_booklet_tg"
            ]
        ]


    # ========================================================
    # 20. Total H&S
    # ========================================================

    hs_detail = project_detail_long.loc[
        project_detail_long["brand"]
        .astype(str)
        .str.upper()
        .eq("H&S")
    ].copy()

    total_hs = create_summary_rows(
        source_df=hs_detail,
        summary_brand="H&S",
        summary_project="Total H&S"
    )


    # ========================================================
    # 21. PTN、GAIA Subtotal、Total PTN
    # ========================================================

    ptn_detail = project_detail_long.loc[
        project_detail_long["brand"]
        .astype(str)
        .str.upper()
        .eq("PTN")
    ].copy()

    gaia_project_keys = {
        "PNT 3MM GAIA SHM&CON",
        "PNT 3MM GAIA OIL",
        "PNT 3MM GAIA - OIL"
    }

    gaia_project_keys = {
        key.upper()
        for key in gaia_project_keys
    }

    gaia_detail = ptn_detail.loc[
        ptn_detail["project_key"]
        .isin(gaia_project_keys)
    ].copy()

    total_gaia = create_summary_rows(
        source_df=gaia_detail,
        summary_brand="PTN",
        summary_project="PNT 3MM GAIA"
    )

    total_ptn = create_summary_rows(
        source_df=ptn_detail,
        summary_brand="PTN",
        summary_project="Total PTN"
    )


    # ========================================================
    # 22. Total REJ
    # ========================================================

    rej_detail = project_detail_long.loc[
        project_detail_long["brand"]
        .astype(str)
        .str.upper()
        .eq("REJ")
    ].copy()

    total_rej = create_summary_rows(
        source_df=rej_detail,
        summary_brand="REJ",
        summary_project="Total REJ"
    )


    # ========================================================
    # 23. Grand Total
    # ========================================================
    # Grand Total = Total H&S + Total PTN + Total REJ
    # ========================================================

    brand_total_source = pd.concat(
        [
            total_hs,
            total_ptn,
            total_rej
        ],
        ignore_index=True
    )

    grand_total = (
        brand_total_source
        .groupby(
            [
                "year_month",
                "channel"
            ],
            as_index=False,
            dropna=False
        )
        .agg(
            booklet=(
                "booklet",
                lambda x: x.sum(min_count=1)
            ),
            actual_value=(
                "actual_value",
                "sum"
            )
        )
    )

    grand_total["brand"] = "Total"
    grand_total["ni_project"] = "Total"
    grand_total["project_key"] = "TOTAL"
    grand_total["sos_date"] = pd.NaT
    grand_total["final_tg"] = common_tg

    valid_grand_total = (
        grand_total["booklet"].notna()
        & (grand_total["booklet"] != 0)
    )

    grand_total["vs_booklet_tg"] = np.where(
        valid_grand_total,
        (
            grand_total["actual_value"]
            / grand_total["booklet"]
            - common_tg
        ),
        np.nan
    )

    grand_total = grand_total[
        [
            "year_month",
            "brand",
            "ni_project",
            "project_key",
            "channel",
            "booklet",
            "actual_value",
            "sos_date",
            "final_tg",
            "vs_booklet_tg"
        ]
    ]


    # ========================================================
    # 24. 合并明细和汇总行
    # ========================================================

    mtd_table_long = pd.concat(
        [
            total_hs,
            hs_detail,

            total_ptn,
            ptn_detail,
            total_gaia,

            total_rej,
            rej_detail,

            grand_total
        ],
        ignore_index=True
    )


    # ========================================================
    # 25. 最终Project范围检查
    # ========================================================

    expected_summary_projects = {
        "TOTAL H&S",
        "TOTAL PTN",
        "PNT 3MM GAIA",
        "TOTAL REJ",
        "TOTAL"
    }

    final_detail_keys = set(
        mtd_table_long.loc[
            ~mtd_table_long["ni_project"]
            .astype(str)
            .str.upper()
            .isin(expected_summary_projects),
            "project_key"
        ]
        .dropna()
        .unique()
    )

    unexpected_project_keys = (
        final_detail_keys
        - allowed_project_keys
    )

    if unexpected_project_keys:

        raise ValueError(
            "最终MTD Table出现不在project_sos_df中的Project Key："
            f"{sorted(unexpected_project_keys)}"
        )


    # ========================================================
    # 26. 行排序规则
    # ========================================================

    def get_row_sort(row):

        brand = str(row["brand"]).strip().upper()
        project = str(row["ni_project"]).strip()
        project_upper = project.upper()

        # H&S
        if project_upper == "TOTAL H&S":
            return (1, 0, "")

        if brand == "H&S":
            return (1, 1, project_upper)

        # PTN
        if project_upper == "TOTAL PTN":
            return (2, 0, "")

        if (
            brand == "PTN"
            and project_upper not in [
                "PNT 3MM GAIA",
                "PNT 3MM GAIA SHM&CON",
                "PNT 3MM GAIA OIL",
                "PNT 3MM GAIA - OIL"
            ]
        ):
            return (2, 1, project_upper)

        if project_upper == "PNT 3MM GAIA":
            return (2, 2, "")

        if project_upper == "PNT 3MM GAIA SHM&CON":
            return (2, 3, "")

        if project_upper in [
            "PNT 3MM GAIA OIL",
            "PNT 3MM GAIA - OIL"
        ]:
            return (2, 4, "")

        # REJ
        if project_upper == "TOTAL REJ":
            return (3, 0, "")

        if brand == "REJ":
            return (3, 1, project_upper)

        # Other SOS Projects
        if brand == "OTHER":
            return (4, 1, project_upper)

        # Grand Total
        if (
            brand == "TOTAL"
            and project_upper == "TOTAL"
        ):
            return (99, 0, "")

        return (50, 0, project_upper)


    sort_values = mtd_table_long.apply(
        get_row_sort,
        axis=1
    )

    mtd_table_long["_section_sort"] = (
        sort_values.map(
            lambda x: x[0]
        )
    )

    mtd_table_long["_row_sort"] = (
        sort_values.map(
            lambda x: x[1]
        )
    )

    mtd_table_long["_project_sort"] = (
        sort_values.map(
            lambda x: x[2]
        )
    )


    # ========================================================
    # 27. SOS Date格式
    # ========================================================

    mtd_table_long["sos_date"] = pd.to_datetime(
        mtd_table_long["sos_date"],
        errors="coerce"
    )

    mtd_table_long["sos_date_display"] = (
        mtd_table_long["sos_date"]
        .dt.strftime("%Y-%m-%d")
        .fillna("")
    )


    # ========================================================
    # 28. Pivot为最终宽表
    # ========================================================

    index_columns = [
        "_section_sort",
        "_row_sort",
        "_project_sort",
        "brand",
        "ni_project",
        "sos_date_display",
        "channel"
    ]

    value_columns = [
        "booklet",
        "actual_value",
        "vs_booklet_tg"
    ]

    mtd_table_output = (
        mtd_table_long
        .set_index(index_columns)[value_columns]
        .unstack("channel")
        .swaplevel(
            0,
            1,
            axis=1
        )
    )


    # ========================================================
    # 29. Channel及Metric顺序
    # ========================================================

    channel_order = [
        "eCom B2C",
        "Dcom",
        "O&O",
        "D-Partner",
        "Total"
    ]

    raw_metric_order = [
        "booklet",
        "actual_value",
        "vs_booklet_tg"
    ]

    ordered_columns = [
        (channel, metric)
        for channel in channel_order
        for metric in raw_metric_order
        if (
            channel,
            metric
        ) in mtd_table_output.columns
    ]

    mtd_table_output = (
        mtd_table_output[
            ordered_columns
        ]
        .reset_index()
    )


    # ========================================================
    # 30. 按辅助列排序
    # ========================================================

    section_sort_col = next(
        col
        for col in mtd_table_output.columns
        if col[0] == "_section_sort"
    )

    row_sort_col = next(
        col
        for col in mtd_table_output.columns
        if col[0] == "_row_sort"
    )

    project_sort_col = next(
        col
        for col in mtd_table_output.columns
        if col[0] == "_project_sort"
    )

    mtd_table_output = (
        mtd_table_output
        .sort_values(
            [
                section_sort_col,
                row_sort_col,
                project_sort_col
            ]
        )
        .reset_index(drop=True)
        .drop(
            columns=[
                section_sort_col,
                row_sort_col,
                project_sort_col
            ]
        )
    )


    # ========================================================
    # 31. 最终Column名称
    # ========================================================

    metric_name_mapping = {
        "booklet": "Booklet",
        "actual_value": actual_column_name,
        "vs_booklet_tg": "vs. Booklet TG"
    }

    renamed_columns = []

    for col in mtd_table_output.columns:

        first = col[0]
        second = col[1]

        if first in [
            "brand",
            "ni_project",
            "sos_date_display"
        ]:

            if first == "sos_date_display":

                renamed_columns.append(
                    ("sos_date", "")
                )

            else:

                renamed_columns.append(
                    (first, "")
                )

        else:

            renamed_columns.append(
                (
                    first,
                    metric_name_mapping.get(
                        second,
                        second
                    )
                )
            )

    mtd_table_output.columns = (
        pd.MultiIndex.from_tuples(
            renamed_columns
        )
    )


    # ========================================================
    # 32. 最终显示格式
    # ========================================================

    booklet_cols = [
        col
        for col in mtd_table_output.columns
        if col[1] == "Booklet"
    ]

    actual_cols = [
        col
        for col in mtd_table_output.columns
        if col[1] == actual_column_name
    ]

    tg_cols = [
        col
        for col in mtd_table_output.columns
        if col[1] == "vs. Booklet TG"
    ]

    # Booklet及Actual保留1位小数
    for col in booklet_cols + actual_cols:

        mtd_table_output[col] = (
            pd.to_numeric(
                mtd_table_output[col],
                errors="coerce"
            )
            .round(1)
        )

    # TG显示成整数百分比
    for col in tg_cols:

        mtd_table_output[col] = (
            pd.to_numeric(
                mtd_table_output[col],
                errors="coerce"
            )
            .map(
                lambda x: (
                    f"{x:.0%}"
                    if pd.notna(x)
                    else ""
                )
            )
        )


    # ========================================================
    # 33. 输出运行状态
    # ========================================================

    print(f"Current month: {current_month}")

    print(
        f"Past / Total FCST Day: "
        f"{current_pastday} / "
        f"{current_total_fcst_day}"
    )

    print(f"Tracking type: {tracking_type}")
    print(f"Actual column: {actual_column_name}")
    print("Actual unit: MM")
    print(f"Common TG: {common_tg:.0%}")

    print(
        f"Detail projects from project_sos_df: "
        f"{len(allowed_project_keys)}"
    )


    return (
        mtd_table_output,
        mtd_table_long
    )

## FYTD

# --Output--

## MTD

In [0]:
# ============================================================
# 调用MTD Table Function
# ============================================================
# 只需要输入：
# 1. current_month：当前Tracking月份，格式为yyyy-mm
# 2. current_pastday：当前月已经过去的FCST Day
# ============================================================

mtd_table_output, mtd_table_long = create_mtd_table(
    current_month=current_month,
    # current_month = "2026-07"

    current_pastday=current_pastday
    # current_pastday = 19.5
)

display(mtd_table_output)

In [0]:
import pandas as pd
import numpy as np


def prepare_mtd_table_for_download(mtd_table_output):
    """
    将mtd_table_output整理成两层Header的下载表。

    Header结构
    ----------
    第一行：
        Channel | SOS | eCom B2C | eCom B2C | eCom B2C | Dcom | ...

    第二行：
        NI Project | SOS | Booklet | MTD + OO |
        vs. Booklet TG | Booklet | ...

    处理内容
    --------
    1. 删除brand列
    2. ni_project显示为NI Project
    3. sos_date显示为SOS
    4. 保留Channel × Metric两层Header
    5. Booklet及MTD / MTD + OO保留1位小数
    6. vs. Booklet TG显示整数百分比
    """

    download_df = mtd_table_output.copy()

    # ========================================================
    # 1. 删除Brand列
    # ========================================================

    brand_columns = [
        col
        for col in download_df.columns
        if (
            col == "brand"
            or (
                isinstance(col, tuple)
                and str(col[0]).strip() == "brand"
            )
        )
    ]

    if brand_columns:
        download_df = download_df.drop(
            columns=brand_columns
        )


    # ========================================================
    # 2. 重新建立两层Column Header
    # ========================================================

    new_columns = []

    for col in download_df.columns:

        if isinstance(col, tuple):

            first_level = str(col[0]).strip()

            second_level = (
                str(col[1]).strip()
                if len(col) > 1
                else ""
            )

            # -----------------------------------------------
            # Project列
            # 第一层：Channel
            # 第二层：NI Project
            # -----------------------------------------------

            if first_level == "ni_project":

                new_columns.append(
                    ("Channel", "NI Project")
                )


            # -----------------------------------------------
            # SOS列
            # 第一层：SOS
            # 第二层：SOS
            # -----------------------------------------------

            elif first_level in [
                "sos_date",
                "sos_date_display"
            ]:

                new_columns.append(
                    ("SOS", "SOS")
                )


            # -----------------------------------------------
            # Channel × Metric列保持不变
            # -----------------------------------------------

            else:

                new_columns.append(
                    (
                        first_level,
                        second_level
                    )
                )

        else:

            column_name = str(col).strip()

            if column_name == "ni_project":

                new_columns.append(
                    ("Channel", "NI Project")
                )

            elif column_name in [
                "sos_date",
                "sos_date_display"
            ]:

                new_columns.append(
                    ("SOS", "SOS")
                )

            else:

                new_columns.append(
                    (column_name, "")
                )


    download_df.columns = pd.MultiIndex.from_tuples(
        new_columns
    )


    # ========================================================
    # 3. 找到Project和SOS列
    # ========================================================

    project_col = (
        "Channel",
        "NI Project"
    )

    sos_col = (
        "SOS",
        "SOS"
    )


    # ========================================================
    # 4. SOS只保留yyyy-mm-dd
    # ========================================================

    if sos_col in download_df.columns:

        download_df[sos_col] = (
            pd.to_datetime(
                download_df[sos_col],
                errors="coerce"
            )
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )


    # ========================================================
    # 5. 找出Booklet、Actual及TG列
    # ========================================================

    booklet_columns = [
        col
        for col in download_df.columns
        if (
            isinstance(col, tuple)
            and len(col) > 1
            and col[1] == "Booklet"
        )
    ]

    actual_columns = [
        col
        for col in download_df.columns
        if (
            isinstance(col, tuple)
            and len(col) > 1
            and col[1] in [
                "MTD",
                "MTD + OO"
            ]
        )
    ]

    tg_columns = [
        col
        for col in download_df.columns
        if (
            isinstance(col, tuple)
            and len(col) > 1
            and col[1] == "vs. Booklet TG"
        )
    ]


    # ========================================================
    # 6. Booklet和Actual保留1位小数
    # ========================================================

    for col in booklet_columns + actual_columns:

        download_df[col] = (
            pd.to_numeric(
                download_df[col],
                errors="coerce"
            )
            .round(1)
        )


    # ========================================================
    # 7. vs. Booklet TG转成整数百分比
    # ========================================================
    # 同时兼容：
    # 0.12
    # "12%"
    # ========================================================

    for col in tg_columns:

        original_values = download_df[col]

        cleaned = (
            original_values
            .astype("string")
            .str.strip()
        )

        already_percentage = (
            cleaned
            .str.contains(
                "%",
                regex=False,
                na=False
            )
        )

        # 只将没有百分号的值转成Numeric
        numeric_values = pd.to_numeric(
            cleaned.where(
                ~already_percentage
            ),
            errors="coerce"
        )

        formatted_values = numeric_values.map(
            lambda x: (
                f"{x:.0%}"
                if pd.notna(x)
                else ""
            )
        )

        download_df[col] = np.where(
            already_percentage,
            cleaned.fillna(""),
            formatted_values
        )


    # ========================================================
    # 8. 确保Project和SOS放在最前面
    # ========================================================

    first_columns = [
        col
        for col in [
            project_col,
            sos_col
        ]
        if col in download_df.columns
    ]

    remaining_columns = [
        col
        for col in download_df.columns
        if col not in first_columns
    ]

    download_df = download_df[
        first_columns
        + remaining_columns
    ]


    # ========================================================
    # 9. 清理Project和SOS缺失值
    # ========================================================

    for col in first_columns:

        download_df[col] = (
            download_df[col]
            .astype("string")
            .fillna("")
            .astype(str)
        )


    return download_df

In [0]:
mtd_download_df = prepare_mtd_table_for_download(
    mtd_table_output
)

display(mtd_download_df)

In [ ]:
def save_csv(
    download_df,
    file_name="MTD_Tracking_2026-07.csv",
):
    """Save the report beside the local notebook using Excel-friendly UTF-8."""
    output_path = Path.cwd() / file_name
    download_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"CSV saved to: {output_path}")
    return output_path

In [ ]:
mtd_download_df = prepare_mtd_table_for_download(
    mtd_table_output
)

csv_path = save_csv(
    download_df=mtd_download_df,
    file_name="MTD_Tracking_2026-07.csv",
)